In [7]:
import torch
from sentence_transformers import SentenceTransformer
from FlagEmbedding import BGEM3FlagModel 
import numpy as np
import time
from ast import literal_eval

from pathlib import Path
import pandas as pd


DATA_PATH = Path("D:/DP/data/storysim_dataset")
titles_df = pd.read_csv(DATA_PATH / "titles.csv")
qa_df = pd.read_csv(DATA_PATH / "cleaned_qa_dataset.csv", converters={"pos_title_ids": literal_eval, "negative_title_ids": literal_eval})
assert not titles_df[titles_df["plot"].isna()].shape[0] > 0
stories = titles_df["plot"].tolist()
queries = qa_df["question"].tolist()
qa_df.describe()

,id
count,3230.000000
mean,1673.153870
std,962.808389
min,0.000000
25%,844.250000
50%,1671.500000
75%,2506.750000
max,3339.000000


In [10]:
model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")
stories_lens = [sum(mask) for mask in model.tokenizer(stories, padding=True, truncation=True, max_length=20000, return_tensors="pt")["attention_mask"]]
queries_lens = [sum(mask) for mask in model.tokenizer(queries, padding=True, truncation=True, max_length=20000, return_tensors="pt")["attention_mask"]]

ValueError: All arrays must be of the same length

In [14]:
query_len_df = pd.DataFrame({"query_len": queries_lens})
story_len_df = pd.DataFrame({"story_len": stories_lens})
query_len_df["query_len"] = query_len_df["query_len"].apply(lambda x: x.item())
story_len_df["story_len"] = story_len_df["story_len"].apply(lambda x: x.item())

display(query_len_df.describe())
display(story_len_df.describe())


,query_len
count,3230.000000
mean,82.939009
std,30.341532
min,29.000000
25%,58.000000
50%,81.000000
75%,103.000000
max,215.000000


,story_len
count,2010.000000
mean,860.612438
std,509.696097
min,28.000000
25%,511.250000
50%,892.000000
75%,1111.000000
max,5539.000000


In [ ]:

models_to_test = {
    "MiniLM-L6-v2": "all-MiniLM-L6-v2",
    "MPNet-base-v2": "all-mpnet-base-v2",
    "BGE-M3": "BAAI/bge-m3",
    "E5": "intfloat/multilingual-e5-large-instruct"
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

def embed_documents(model_name, model_id, stories):
    if model_name == "BGE-M3":
        model = BGEM3FlagModel(model_id, use_fp16=True if device == 'cuda' else False)
        raw_output = model.encode(stories,
                                batch_size=4,
                                max_length=512, 
                                convert_to_numpy=True)
        embeddings = raw_output['dense_vecs']
        return embeddings
    else:
        model = SentenceTransformer(model_id, device=device)
        embeddings = model.encode(stories, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    return embeddings

def get_detailed_instruct(task_description: str, query: str) -> str:
            return f'Instruct: {task_description}\nQuery: {query}'

def embed_queries(model_name, model_id, queries):
    
    if model_name == "E5":
        task = 'Given a story retrieve relevant passages that is similar to the story'

        queries = [
            get_detailed_instruct(task, q)
            for q in queries
        ]
    embeddings = embed_documents(model_name, model_id, queries)
    return embeddings

query_embeddings = {}
plot_embeddings = {}


for model_name, model_id in models_to_test.items():
    print(f"\n--- Processing model: {model_name} ---")
    plot_save_file = DATA_PATH/f'plot_embeddings_{model_name}.npy'
    if plot_save_file.exists():
        embeddings = np.load(plot_save_file)
    else:
        embeddings = embed_documents(model_name, model_id, stories)
        with open(plot_save_file, 'wb') as f:
            np.save(f, embeddings)
    
    query_save_file = DATA_PATH/f'query_embeddings_{model_name}.npy'
    if query_save_file.exists():
        qembeddings = np.load(query_save_file)
    else:
        qembeddings = embed_queries(model_name, model_id, queries)
        with open(query_save_file, 'wb') as f:
            np.save(f, qembeddings)
    
    plot_embeddings[model_name] = embeddings
    query_embeddings[model_name] = qembeddings



In [2]:
def calc_metrics(q_emb, doc_embs, qa_df, titles_df, k=10):
    """
    Calculate retrieval metrics for query embeddings against document embeddings.
    
    Args:
        q_emb: Query embeddings, where q_emb[i] corresponds to qa_df.iloc[i]
        doc_embs: Document embeddings, where doc_embs[i] corresponds to titles_df.iloc[i]
        qa_df: DataFrame containing queries and ground truth, with pos_title_ids column
        titles_df: DataFrame containing document titles (or IDs if matching is done by index)
        k: Maximum number of top documents to consider for metrics (default: 10)
    
    Returns:
        Dictionary of retrieval metrics
    """
    # We'll assume doc_embs is aligned so that doc_embs[i] corresponds to the i-th row in titles_df.
    # Also assume that 'pos_title_ids' contains the string id to the titles_df title column.
    
    # Compute pairwise dot-product similarity (cosine if embeddings are normalized).
    similarities = np.dot(q_emb, doc_embs.T)
    
    # Prepare lists to aggregate metrics across queries
    recall_at_1 = []
    recall_at_5 = []
    recall_at_10 = []
    precision_at_1 = []
    precision_at_5 = []
    precision_at_10 = []
    mrr_list = []
    ap_list = []
    
    for i, row in qa_df.iterrows():
        # The set (or list) of relevant doc indices for this query
        rel_title_ids = row['pos_title_ids'] 
        relevant_docs = []
        for title in rel_title_ids:
            try:
                relevant_docs.append(titles_df[titles_df["title"] == title].index.values[0])
            except:
                continue
        
        # Sort documents in descending order of similarity
        sorted_indices = np.argsort(-similarities[i])
        
        
        # If there are no relevant docs, skip
        if len(relevant_docs) == 0:
            # Append zeros
            recall_at_1.append(0.0)
            recall_at_5.append(0.0)
            recall_at_10.append(0.0)
            precision_at_1.append(0.0)
            precision_at_5.append(0.0)
            precision_at_10.append(0.0)
            mrr_list.append(0.0)
            ap_list.append(0.0)
            continue
        
        # Recall@k = (number of relevant docs in top k) / (total relevant docs)
        # Precision@k = (number of relevant docs in top k) / k
        # We'll do this specifically for k=1, k=5, k=10.
        def recall_at(n):
            return len(set(sorted_indices[:n]).intersection(relevant_docs)) / len(relevant_docs)
        
        def precision_at(n):
            return len(set(sorted_indices[:n]).intersection(relevant_docs)) / float(n)
        
        recall_1 = recall_at(1)
        recall_5 = recall_at(5)
        recall_10 = recall_at(10)
        
        precision_1 = precision_at(1)
        precision_5 = precision_at(5)
        precision_10 = precision_at(10)
        
        # Calculate MRR (Mean Reciprocal Rank)
        # Find the first relevant doc in the sorted list
        first_relevant_rank = None
        for rank_idx, doc_idx in enumerate(sorted_indices):
            if doc_idx in relevant_docs:
                first_relevant_rank = rank_idx + 1  # +1 for 1-based rank
                break
        mrr = 1.0 / first_relevant_rank if first_relevant_rank else 0.0
        
        # Calculate Average Precision (AP) for the top k (or entire set)
        # For MAP, we'll compute AP up to k (or you could do the entire ranking).
        ap_temp = 0.0
        correct_count = 0
        for rank_idx, doc_idx in enumerate(sorted_indices[:k]):
            if doc_idx in relevant_docs:
                correct_count += 1
                ap_temp += correct_count / float(rank_idx + 1)
        ap = ap_temp / len(relevant_docs) if relevant_docs else 0.0
        
        # Store values
        recall_at_1.append(recall_1)
        recall_at_5.append(recall_5)
        recall_at_10.append(recall_10)
        precision_at_1.append(precision_1)
        precision_at_5.append(precision_5)
        precision_at_10.append(precision_10)
        mrr_list.append(mrr)
        ap_list.append(ap)
    
    # Aggregate metrics
    results = {
        'recall@5': np.mean(recall_at_5),
        'recall@10': np.mean(recall_at_10),
        'precision@1': np.mean(precision_at_1),
        'precision@5': np.mean(precision_at_5),
        'precision@10': np.mean(precision_at_10),
        'mrr': np.mean(mrr_list),
        'map': np.mean(ap_list)
    }
    
    return results


In [ ]:
# Calculate metrics for all models
results = {}
for model_name in models_to_test.keys():
    print(f"\nEvaluating {model_name}...")
    metrics = calc_metrics(
        query_embeddings[model_name],
        plot_embeddings[model_name],
        qa_df,
        titles_df
    )
    results[model_name] = metrics
    print(f"  Recall@5: {metrics['recall@5']:.4f}")
    print(f"  Recall@10: {metrics['recall@10']:.4f}")
    print(f"  MRR: {metrics['mrr']:.4f}")

# Create results dataframe
results_df = pd.DataFrame(results).T
results_df.to_csv(DATA_PATH/"embed_comparison.csv")
print("\nModel Comparison:")
display(results_df)


Evaluating MiniLM-L6-v2...
  Recall@5: 0.4214
  Recall@10: 0.5073
  MRR: 0.5043

Evaluating MPNet-base-v2...
  Recall@5: 0.4878
  Recall@10: 0.5709
  MRR: 0.5657

Evaluating BGE-M3...
  Recall@5: 0.5152
  Recall@10: 0.6207
  MRR: 0.5980

Evaluating E5...
  Recall@5: 0.7204
  Recall@10: 0.8048
  MRR: 0.7779

Model Comparison:


,recall@1,recall@5,recall@10,precision@1,precision@5,precision@10,mrr,map
MiniLM-L6-v2,0.191150,0.421427,0.507288,0.403096,0.187183,0.112601,0.504254,0.349925
MPNet-base-v2,0.219135,0.487844,0.570872,0.461300,0.216285,0.127307,0.565672,0.404337
BGE-M3,0.233741,0.515242,0.620672,0.488545,0.226068,0.137059,0.597980,0.431760
E5,0.337700,0.720367,0.804845,0.692260,0.317709,0.179009,0.777891,0.639122


In [4]:
del query_embeddings
del plot_embeddings

In [ ]:




e5_instructions = {
    "setting_retrieval": "Given a story description, retrieve similar stories with similar plot, characters, setting and themes",
    "character_dynamics": "Find stories with comparable character relationships and interactions based on the described personalities and roles",
    "thematic_retrieval": "Retrieve stories that explore similar themes and moral dilemmas as those presented in the query",
    "plot_development": "Identify stories with analogous narrative structures, key events, and plot progression",
    "genre_specific": "Locate stories within the same genre that share common tropes and storytelling conventions",
    "mood_based": "Find stories that match the emotional tone and atmosphere described in the query",
    "cross_genre": "Discover stories from different genres that share similar narrative elements or plot devices",
    "continuation_search": "Retrieve stories that could serve as plausible sequels or prequels to the described narrative",
    "adaptation_candidates": "Identify stories with adaptation potential matching the described format and audience"
}

results = {}
plot_embeddings = np.load('D:/DP/data/storysim_dataset/plot_embeddings_E5.npy')
for task, instruction in e5_instructions.items():
    print(f"\n--- Processing task: {task} ---")
    save_file = DATA_PATH/f'e5_instructions_{task}.npy'
    if save_file.exists():
        embeddings = np.load(save_file)
    else:
        task_queries = [get_detailed_instruct(instruction, q) for q in queries]
        embeddings = embed_documents("E5", "intfloat/multilingual-e5-large-instruct", task_queries)
        with open(save_file, 'wb') as f:
            np.save(f, embeddings)
   
    results[task] = calc_metrics(embeddings, plot_embeddings, qa_df, titles_df)

results_instructions_df = pd.DataFrame(results).T
print("\nInstructions Comparison:")
display(results_instructions_df)

results_instructions_df.to_csv(DATA_PATH / "embed_e5_instruct_results.csv")




--- Processing task: setting_retrieval ---

--- Processing task: character_dynamics ---

--- Processing task: thematic_retrieval ---

--- Processing task: plot_development ---

--- Processing task: genre_specific ---

--- Processing task: mood_based ---

--- Processing task: cross_genre ---

--- Processing task: continuation_search ---

--- Processing task: adaptation_candidates ---


Batches:   0%|          | 0/101 [00:00<?, ?it/s]


Instructions Comparison:


,recall@1,recall@5,recall@10,precision@1,precision@5,precision@10,mrr,map
setting_retrieval,0.330038,0.711138,0.800520,0.687307,0.313746,0.178080,0.773335,0.630234
character_dynamics,0.343206,0.730533,0.816470,0.713932,0.322539,0.181827,0.793448,0.655542
thematic_retrieval,0.339844,0.720152,0.808954,0.705573,0.317523,0.179876,0.785899,0.645347
plot_development,0.254766,0.586596,0.719676,0.539319,0.260062,0.160279,0.648522,0.499153
genre_specific,0.220143,0.538202,0.673633,0.473684,0.239195,0.150402,0.597019,0.445217
mood_based,0.294297,0.658526,0.758398,0.620433,0.290898,0.168854,0.717542,0.567325
cross_genre,0.194436,0.492757,0.622419,0.416409,0.219752,0.139628,0.547881,0.397268
continuation_search,0.325770,0.717498,0.809417,0.678638,0.316347,0.180000,0.769839,0.631503
adaptation_candidates,0.098669,0.330942,0.479861,0.207740,0.148978,0.108019,0.358348,0.240311


In [6]:
results_df.to_csv(DATA_PATH/"embed_comparison.csv")
results_instructions_df.to_csv(DATA_PATH / "embed_e5_instruct_results.csv")